# Tech layoffs activity calendar

Explore reported tech layoff events day by day. Each square is one day; darker red means more reported layoffs or more recorded events, depending on the selected metric.

Data source: [Layoffs.fyi](https://layoffs.fyi/). This app uses the bundled `layoffs.csv` snapshot; individual event sources are included in its `Source` column.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import mercury as mr


csv_candidates = [Path("layoffs.csv"), Path("layoffs/layoffs.csv")]
csv_path = next((path for path in csv_candidates if path.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Could not find layoffs.csv in the notebook or repository directory.")

layoffs = pd.read_csv(csv_path)
layoffs["Date"] = pd.to_datetime(layoffs["Date"], errors="coerce").dt.normalize()
layoffs["# Laid Off"] = pd.to_numeric(layoffs["# Laid Off"], errors="coerce")
layoffs = layoffs.dropna(subset=["Date"]).copy()

In [ ]:
first_date = layoffs["Date"].min().date().isoformat()
last_date = layoffs["Date"].max().date().isoformat()

country_choices = ["All countries"] + sorted(layoffs["Country"].dropna().unique().tolist())
industry_choices = ["All industries"] + sorted(layoffs["Industry"].dropna().unique().tolist())
stage_choices = ["All stages"] + sorted(layoffs["Stage"].dropna().unique().tolist())

In [ ]:
date_range = mr.DateRange(
    label="Layoff date range",
    value=[first_date, last_date],
    min=first_date,
    max=last_date,
    start_url_key="from",
    end_url_key="to",
)

In [ ]:
country_filter = mr.Select(
    label="Country",
    value="All countries",
    choices=country_choices,
    url_key="country",
)

In [ ]:
industry_filter = mr.Select(
    label="Industry",
    value="All industries",
    choices=industry_choices,
    url_key="industry",
)

In [ ]:
stage_filter = mr.Select(
    label="Company stage",
    value="All stages",
    choices=stage_choices,
    url_key="stage",
)

In [ ]:
metric_filter = mr.Select(
    label="Calendar metric",
    value="Employees laid off (reported)",
    choices=["Employees laid off (reported)", "Recorded layoff events"],
    url_key="metric",
)

In [ ]:
scale_filter = mr.Select(
    label="Color scaling",
    value="Square root (more contrast)",
    choices=["Linear (raw)", "Square root (more contrast)", "Log (strong contrast)"],
    url_key="scale",
)

In [ ]:
color_filter = mr.Select(
    label="Calendar color",
    value="green",
    choices=["green", "red"],
    url_key="color",
)

In [ ]:
selected_start = pd.Timestamp(date_range.value[0] or first_date)
selected_end = pd.Timestamp(date_range.value[1] or last_date)

filtered = layoffs[layoffs["Date"].between(selected_start, selected_end)].copy()

if country_filter.value != "All countries":
    filtered = filtered[filtered["Country"] == country_filter.value]

if industry_filter.value != "All industries":
    filtered = filtered[filtered["Industry"] == industry_filter.value]

if stage_filter.value != "All stages":
    filtered = filtered[filtered["Stage"] == stage_filter.value]

if metric_filter.value == "Employees laid off (reported)":
    daily = (
        filtered.groupby("Date", as_index=False)["# Laid Off"]
        .sum()
        .rename(columns={"Date": "date", "# Laid Off": "value"})
    )
    calendar_title = "Reported employees laid off each day"
    calendar_unit = "employees"
else:
    daily = (
        filtered.groupby("Date").size().rename("value").reset_index()
        .rename(columns={"Date": "date"})
    )
    calendar_title = "Recorded tech layoff events each day"
    calendar_unit = "events"

if daily.empty:
    daily = pd.DataFrame({"date": [selected_start], "value": [0]})

if scale_filter.value == "Square root (more contrast)":
    daily["display_value"] = daily["value"].clip(lower=0) ** 0.5
    calendar_title = f"{calendar_title} · sqrt-scaled"
    calendar_unit = f"sqrt({calendar_unit})"
elif scale_filter.value == "Log (strong contrast)":
    daily["display_value"] = np.log1p(daily["value"].clip(lower=0))
    calendar_title = f"{calendar_title} · log-scaled"
    calendar_unit = f"log1p({calendar_unit})"
else:
    daily["display_value"] = daily["value"]

In [ ]:
known_count_events = filtered["# Laid Off"].notna().sum()
unknown_count_events = filtered["# Laid Off"].isna().sum()

mr.Indicator([
    mr.Indicator(f"{len(filtered):,}", label="Recorded events"),
    mr.Indicator(f"{filtered['# Laid Off'].sum():,.0f}", label="Reported layoffs"),
    mr.Indicator(f"{known_count_events:,}", label="Events with a count"),
    mr.Indicator(f"{unknown_count_events:,}", label="Events without a count"),
])

In [ ]:
mr.ActivityCalendar(
    daily,
    date="date",
    value="display_value",
    title="Tech Layoffs, day by Day: 2020-2026",
    unit=calendar_unit,
    color=color_filter.value,
    start_date=selected_start,
    end_date=selected_end,
)

## Filtered layoff events

The employee total only includes events for which Layoffs.fyi reports a `# Laid Off` value. Events without a reported count remain visible in the event metric and table.

In [ ]:
event_table = filtered.sort_values("Date", ascending=False)[
    ["Date", "Company", "# Laid Off", "Country", "Industry", "Stage", "Source"]
].copy()
event_table["Date"] = event_table["Date"].dt.strftime("%Y-%m-%d")
event_table.head(100)